# 6.4.6 Document Understanding

Layout-aware feature extraction from a synthetic document: bounding boxes, element types, reading order, and IoU matrix.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

Path('output').mkdir(exist_ok=True)

elements = [
    {'type': 'title',  'box': [50, 40, 750, 100]},
    {'type': 'text',   'box': [50, 120, 370, 300]},
    {'type': 'text',   'box': [430, 120, 750, 300]},
    {'type': 'table',  'box': [50, 320, 750, 550]},
    {'type': 'figure', 'box': [50, 570, 370, 780]},
    {'type': 'footer', 'box': [50, 950, 750, 990]},
]

def iou(a, b):
    xi1,yi1 = max(a[0],b[0]), max(a[1],b[1])
    xi2,yi2 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0,xi2-xi1)*max(0,yi2-yi1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-10)

for e in elements:
    x1,y1,x2,y2 = e['box']
    print(f"{e['type']:8s}: area={(x2-x1)*(y2-y1):6.0f} aspect_ratio={(x2-x1)/(y2-y1+1e-10):.2f}")

In [ ]:
n = len(elements)
iou_mat = np.array([[iou(a['box'],b['box']) for b in elements] for a in elements])
COLORS = {'title':'#e74c3c','text':'#3498db','table':'#2ecc71','figure':'#9b59b6','footer':'#95a5a6'}
PAGE_H = 1000

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for e in elements:
    x1,y1,x2,y2 = e['box']
    c = COLORS.get(e['type'], '#bdc3c7')
    axes[0].add_patch(Rectangle((x1, PAGE_H-y2), x2-x1, y2-y1, linewidth=1.5, edgecolor=c, facecolor=c, alpha=0.3))
    axes[0].text((x1+x2)/2, PAGE_H-(y1+y2)/2, e['type'], ha='center', va='center', fontsize=8)
axes[0].set(xlim=(0,800), ylim=(0,1000), title='Document Layout'); axes[0].set_aspect('equal')

im = axes[1].imshow(iou_mat, cmap='Blues', vmin=0, vmax=0.3)
axes[1].set(title='IoU Matrix', xticks=range(n), yticks=range(n),
            xticklabels=[e['type'][:4] for e in elements], yticklabels=[e['type'][:4] for e in elements])
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('output/document_understanding.png')
print('Saved document_understanding.png')